# 단어 벡터 연산

In [ ]:
import gensim.downloader as api

model = api.load("glove-wiki-gigaword-50")

In [ ]:
print(model.similarity("cat", "dog"), model.similarity("cat", "car"))
print(model.most_similar("computer", topn=10))
print(model.doesnt_match(["apple", "banana", "orange", "computer"]))

# 데이터셋

In [ ]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

In [ ]:
encoded = tokenizer.encode("Hello world, Hello tokenizer")
print(encoded)
for token in encoded:
    print("■", tokenizer.decode([token]))
print(tokenizer.decode(encoded))

In [ ]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

print(raw_text[:300])

In [ ]:
encoded = tokenizer.encode(raw_text)
print(len(encoded))

In [ ]:
context_size = 4
input_chunk = encoded[:context_size]
target_chunk = encoded[1:context_size+1]
print(input_chunk, target_chunk)

In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset
import tiktoken

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        encoded = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        self.input_ids = []
        self.target_ids = []

        assert len(encoded) > max_length

        for i in range(0, len(encoded) - max_length, stride):
            input_chunk = encoded[i : i + max_length]
            target_chunk = encoded[i + 1 : i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

def create_dataloader_v1(txt, batch_size=4, max_length=256, stride=128, shuffle=True, drop_last=True, num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers)


dataloader = create_dataloader_v1(raw_text, batch_size=1, max_length=4, stride=1, shuffle=False)

data_iter = iter(dataloader)
print(next(data_iter))
print(next(data_iter))
print(next(data_iter))

In [ ]:
import torch
import torch.nn as nn

input_ids = torch.tensor([0, 5,])

vocab_size = 6
output_dim = 3

emb_layer = nn.Embedding(vocab_size, output_dim)
print(emb_layer.weight)

token_embs = emb_layer(input_ids)
print(token_embs)

In [ ]:
vocab_size = 50257
output_dim = 256

token_embedding_layer = nn.Embedding(vocab_size, output_dim)

max_length = 4
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=max_length, stride=max_length, shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)

print("inputs", inputs.shape)
token_embeddings = token_embedding_layer(inputs)
print("token embeddings", token_embeddings.shape)

context_length = max_length
pos_embedding_layer = nn.Embedding(context_length, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(max_length)) # TODO: 헷갈림 주의. 입력으로 [0, 1, 2, 3] 벡터
print("pos embeddings", pos_embeddings.shape)

input_embeddings = token_embeddings + pos_embeddings
print("final input embeddings", input_embeddings.shape)

# Attention

In [ ]:
import torch
import torch.nn as nn

class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        queries = self.W_query(x) # [b, num_tokens, d_in] @ [d_in, d_out] = [b, num_tokens, d_out]
        keys = self.W_key(x) # [b, num_tokens, d_in] @ [d_in, d_out] = [b, num_tokens, d_out]
        values = self.W_value(x) # [b, num_tokens, d_in] @ [d_in, d_out] = [b, num_tokens, d_out]

        attn_score = queries @ keys.transpose(1, 2)
        attn_score.masked_fill_(self.mask.bool()[:num_tokens,:num_tokens], -torch.inf)
        attn_score = attn_score / keys.shape[-1] ** 0.5
        attn_weight = torch.softmax(attn_score, dim=-1)
        attn_weight = self.dropout(attn_weight) # [b, num_tokens, num_tokens]
        context_vec = attn_weight @ values
        return context_vec


inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

d_in = inputs.shape[1]   # 입력 차원 (d=3)
d_out = 2                # Q, K, V의 출력 차원 (d=2)

batch = torch.stack((inputs, inputs), dim=0)

torch.manual_seed(123)
context_length = batch.shape[1] 
print("context_length:", context_length)

ca = CausalAttention(d_in, d_out, context_length, 0.0)
context_vecs = ca(batch)

print(context_vecs)

In [ ]:
class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList(
            [CausalAttention(d_in, d_out, context_length, dropout, qkv_bias)
            for _ in range(num_heads)]
        )

    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)


torch.manual_seed(123)
context_length = batch.shape[1] 
d_in, d_out = 3, 2

mha = MultiHeadAttentionWrapper(d_in, d_out, context_length, 0.0, num_heads=3)
context_vecs = mha(batch)

print(context_vecs)

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        queries = self.W_query(x) # [b, num_tokens, d_out]
        keys = self.W_key(x) # [b, num_tokens, d_out]
        values = self.W_value(x) # [b, num_tokens, d_out]

        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim) # [b, num_tokens, num_heads, head_dim]
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim) # [b, num_tokens, num_heads, head_dim]
        values = values.view(b, num_tokens, self.num_heads, self.head_dim) # [b, num_tokens, num_heads, head_dim]

        queries = queries.transpose(1, 2) # [b, num_heads, num_tokens, head_dim]
        keys = keys.transpose(1, 2) # [b, num_heads, num_tokens, head_dim]
        values = values.transpose(1, 2) # [b, num_heads, num_tokens, head_dim]

        attn_score = queries @ keys.transpose(2, 3) # [b, num_heads, num_tokens, num_tokens]
        attn_score.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
        attn_score = attn_score / self.head_dim ** 0.5
        attn_weight = torch.softmax(attn_score, dim=-1) # [b, num_heads, num_tokens, num_tokens]
        attn_weight = self.dropout(attn_weight)

        context_vecs = attn_weight @ values # [b, num_heads, num_tokens, head_dim]
        context_vecs = context_vecs.transpose(1, 2) # [b, num_tokens, num_heads, head_dim]
        context_vecs = context_vecs.flatten(2, 3)  # [b, num_tokens, d_out]

        return self.W_out_proj(context_vecs)

torch.manual_seed(123)

batch_size, context_length, d_in = batch.shape
d_out = 2 
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, 2)
context_vecs = mha(batch)

print(context_vecs)